# Fine-Tuning di Llama 3.1 8B con LoRA (PEFT)
Questo notebook esegue il fine-tuning controllato (SFT) di un Large Language Model utilizzando la tecnica **LoRA (Low-Rank Adaptation)** al fine di realizzare **Information Extraction**. Nello specifico abbiamo due sotto-task simultanei: **Named Entity Recognition (NER)** per identificare i termini e **Relation Extraction (Estrazione di Relazioni)** per capire come sono legati tra loro, ovvero per l'estrazione di triple (es. Vitello Tonnato | USA_INGREDIENTE | Tonno sott'olio)

In [1]:
!pip install -U bitsandbytes transformers accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 102.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.2 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fo

In [2]:
# ===================================
# 1. IPERPARAMETRI E CONFIGURAZIONI 
# ===================================
import os
import torch
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer,  DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
N_EPOCHS = 3
LR = 2e-4  # Adattato per Causal LM rispetto a BERT
MAX_LEN = 512

# Usiamo una versione pre-quantizzata a 4-bit per far girare Llama 3.1 8B gratis sulla T4 di Kaggle
MODEL_CHECKPOINT = "unsloth/meta-llama-3.1-8b-bnb-4bit" 
DATASET_FILE = "/kaggle/input/datasets/paolaapap/dataset-triple-culinarie/dataset_triple_culinarie.jsonl"

gc.collect()
torch.cuda.empty_cache()

print("📥 Caricamento del dataset generato dal ChromaDB...")
dataset = load_dataset("json", data_files=DATASET_FILE)["train"]

# Dividiamo in Train e Validation set 
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
val_dataset = dataset_split["test"]

print(f"📊 Dataset pronto. Train: {len(train_dataset)} esempi, Validation: {len(val_dataset)} esempi.")

📥 Caricamento del dataset generato dal ChromaDB...


Generating train split: 0 examples [00:00, ? examples/s]

📊 Dataset pronto. Train: 315 esempi, Validation: 35 esempi.


In [3]:
# =========================================================================
# 2. TOKENIZZAZIONE E FORMATTAZIONE PROMPT
# =========================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
tokenizer.pad_token = tokenizer.eos_token  # Gestione del token di fine stringa come padding
tokenizer.padding_side = "right"

def format_and_tokenize_function(examples):
    """Concatena istruzione, input e output nel formato che il modello deve imparare."""
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        # Creiamo la struttura del testo (SFT - Supervised Fine-Tuning)
        text = f"### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Output:\n{out}{tokenizer.eos_token}"
        texts.append(text)
        
    # Tokenizziamo troncando alla lunghezza massima per la memoria della GPU
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding=False)
    
    # Nelle architetture Causal LM, le label per calcolare la Loss corrispondono agli input_ids stessi
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("🧼 Esecuzione della Tokenizzazione dei dati...")
tokenized_train = train_dataset.map(format_and_tokenize_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(format_and_tokenize_function, batched=True, remove_columns=val_dataset.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

🧼 Esecuzione della Tokenizzazione dei dati...


Map:   0%|          | 0/315 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

In [4]:
# =========================================================================
# 3. LORA CONFIGURATION (Speculare alla Cella 50 del Professore)
# =========================================================================
lora_config = LoraConfig(
    r=8,                                # Il Rango (Rank) delle matrici sottili LoRA
    lora_alpha=16,                      # Fattore di scala costante matematico
    target_modules=["q_proj", "v_proj"],# Bersagliamo Query e Value di Llama
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM        # Impostiamo Causal LM invece di classificazione sequenze
)

print("🤖 Caricamento del Modello Base e inizializzazione dell'adattatore PEFT...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_CHECKPOINT, 
    torch_dtype=torch.float16,
    device_map="auto"
)

# Applichiamo i parametri LoRA sul modello originale congelato (Cella 51 del Prof)
base_model = prepare_model_for_kbit_training(base_model)
base_model.config.use_cache = False

model_lora = get_peft_model(base_model, lora_config)
model_lora.print_trainable_parameters()

🤖 Caricamento del Modello Base e inizializzazione dell'adattatore PEFT...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


In [ ]:
# =========================================================================
# 4. TRAINING ARGUMENTS E TRAINER (Pulito e Aggiornato)
# =========================================================================
training_args = TrainingArguments(
    output_dir="./culinary_lora_output",
    learning_rate=LR,
    per_device_train_batch_size=1,       
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,  
    gradient_checkpointing_kwargs={"use_reentrant": False},
    num_train_epochs=N_EPOCHS,
    eval_strategy="epoch",               
    save_strategy="epoch",               
    # 💡 Rimosso logging_dir per eliminare il DeprecationWarning
    logging_strategy="steps",
    logging_steps=10,
    fp16=True, 
    optim="paged_adamw_8bit",
    save_total_limit=1,
    report_to="none"                     
)

# Inizializziamo il modulo Trainer di Hugging Face
trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("🏋️ Avvio del processo di Fine-Tuning LoRA...")
trainer.train()

🏋️ Avvio del processo di Fine-Tuning LoRA...


Epoch,Training Loss,Validation Loss


In [ ]:
# =========================================================================
# 5. SALVATAGGIO DELL'ADATTATORE FINALE
# =========================================================================
OUTPUT_ADAPTER_DIR = "./final_culinary_lora_adapter"
model_lora.save_pretrained(OUTPUT_ADAPTER_DIR)
tokenizer.save_pretrained(OUTPUT_ADAPTER_DIR)

print(f"\n🎉 Addestramento concluso con successo! I pesi LoRA leggeri sono salvati in: {OUTPUT_ADAPTER_DIR}")
print("Scarica questa cartella per procedere all'integrazione locale.")

In [6]:
# =========================================================================
# 6. ESPORTAZIONE PER OLLAMA (GGUF) TRAMITE UNSLOTH
# =========================================================================

# 1. Installiamo Unsloth (solo per l'esportazione)
#!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
#!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

from unsloth import FastLanguageModel

# 2. Sostituisci "OUTPUT_ADAPTER_DIR" con il percorso reale dove 
# il tuo script ha appena salvato l'adapter (es. "./modello_finito")
percorso_adapter = "./final_culinary_lora_adapter"

print("Caricamento del modello e dell'adapter per la conversione...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = percorso_adapter, 
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 3. Fonde il modello base e l'adapter e lo quantizza a 4-bit per Ollama
print("Esportazione in formato GGUF in corso. Potrebbe volerci qualche minuto...")
model.save_pretrained_gguf(
    "modello_culinario_ollama", 
    tokenizer, 
    quantization_method = "q4_k_m" # Il miglior compromesso velocità/qualità
)
print("Esportazione completata! Troverai il file .gguf nella cartella 'modello_culinario_ollama'.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\xolor\Miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0605 17:51:33.515000 26876 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0605 17:51:33.575000 26876 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[bitsandbytes.cextension|ERROR]bitsandbytes library load error: Configured CUDA binary not found at c:\Users\xolor\Miniconda3\Lib\site-packages\bitsandbytes\libbitsa

Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


Caricamento del modello e dell'adapter per la conversione...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5070. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0+cu132. CUDA: 12.0. CUDA Toolkit: 13.2. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 1696.38it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:24<00:00, 21.13s/it]


Unsloth: Merge process complete. Saved to `c:\Users\xolor\OneDrive - Università degli Studi di Catania\Università\Magistrale\Cognitive\Progetto\modello_culinario_ollama`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: llama.cpp folder exists but binaries not found - will build
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['modello_culinario_ollama_gguf\\Meta-Llama-3.1-8B.BF16.g

In [7]:
import os

llama_path = r"C:\Users\xolor\.unsloth\llama.cpp"

print(f"Controllo il percorso: {llama_path}")

if os.path.exists(llama_path):
    print("✅ La cartella principale ESISTE.")
    contenuto = os.listdir(llama_path)
    
    # Stampiamo i primi elementi per capire cosa c'è dentro
    print(f"📁 Contenuto trovato: {contenuto[:5]}...")
    
    if "convert_hf_to_gguf.py" in contenuto:
        print("🎉 PERFETTO! I file sono esattamente dove devono essere. Il problema è un altro.")
    else:
        print("\n❌ ERRORE MATRIOSKA TROVATO!")
        print("I file sorgente non sono qui dentro. Probabilmente sono chiusi dentro una di queste sottocartelle.")
        print("👉 SOLUZIONE: Apri Esplora Risorse, entra nella sottocartella, taglia (CTRL+X) tutti i file e incollali (CTRL+V) direttamente in C:\\Users\\xolor\\.unsloth\\llama.cpp")
else:
    print("❌ La cartella NON ESISTE (forse c'è un errore di battitura nel nome o un problema di permessi).")

Controllo il percorso: C:\Users\xolor\.unsloth\llama.cpp
✅ La cartella principale ESISTE.
📁 Contenuto trovato: ['.clang-format', '.clang-tidy', '.devops', '.dockerignore', '.ecrc']...
🎉 PERFETTO! I file sono esattamente dove devono essere. Il problema è un altro.
